In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week7-assignment-1"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
# /public/trendytech/datasets/cust_transf.csv

In [3]:
# [itv024128@g02 ~]$ hadoop fs -head /public/trendytech/datasets/cust_transf.csv
# 1001,2023-05-15,1001,49.99
# 1002,2023-05-16,1002,29.99
# 1003,2023-05-17,1003,39.99
# 1004,2023-05-18,1004,19.99

In [4]:
## customer ID, purchase date, product id, and amount

In [5]:
# A) Design a caching mechanism using dataframes to enhance theperformance of data retrieval for the following use cases:

In [6]:
# A.1 Your marketing team wants to identify the top-selling products based on revenue for a given time period. 
# The query is expected to be executed frequently, and the results need to be returned quickly. 
# Design a caching strategy that efficiently retrieves the top-selling products by revenue.
# Additionally, demonstrate the impact of caching by comparing the retrievaltime 
# for Top 10 best-selling products from start_date = "2023-05-01" toend_date = "2023-06-08" before and after implementing the caching strategy.
# [Note : Strategize your caching in such a way that the right Dataframes arecached at the right time for maximal performance gains]

In [2]:
cust_schema = 'customer_id long, purchase_date date, product_id long, amount double'

In [3]:
cust_df = spark.read.schema(cust_schema).csv('/public/trendytech/datasets/cust_transf.csv')

### AGG functions min and max

In [45]:
cust_df.agg(min("purchase_date"),max('purchase_date')).show()

+------------------+------------------+
|min(purchase_date)|max(purchase_date)|
+------------------+------------------+
|        2023-05-15|        2023-06-15|
+------------------+------------------+



In [9]:
cust_df1 = cust_df.select("purchase_date","product_id","amount").cache()

In [10]:
cust_df2 = cust_df1.filter("purchase_date >= '2023-05-01' and purchase_date <= '2023-06-08' ")

In [11]:
#cust_df3 = cust_df2.groupBy("product_id").agg(sum("amount").alias("Total_sales")).sort("Total_sales",ascending=False)

In [12]:
cust_df3 = cust_df2.groupBy("product_id").sum("amount").withColumnRenamed("sum(amount)","Total_sales").sort("Total_sales",ascending=False)

In [13]:
cust_df3.show(10)

+----------+--------------------+
|product_id|         Total_sales|
+----------+--------------------+
|      1003| 5.725592243903786E8|
|      1001|  5.56682641192824E8|
|      1002| 4.293836243948648E8|
|      1004|2.8620802440276194E8|
|      1005| 2.782856412021384E8|
|      1015|  12537.909999999963|
|      1014|  11492.909999999963|
|      1013|  10447.909999999963|
|      1012|   9402.909999999965|
|      1011|   8357.909999999967|
+----------+--------------------+
only showing top 10 rows



In [14]:
# change dates

In [15]:
cust_df4 = cust_df1.filter("purchase_date >= '2023-05-01' and purchase_date <= '2023-05-15' ")

### AGG functions- sum

In [16]:
cust_df5 = cust_df4.groupBy("product_id").agg(sum("amount").alias("Total_sales")).sort("Total_sales",ascending=False)

In [17]:
cust_df5.show(10) ## runs very fast, as its from cache

+----------+-------------------+
|product_id|        Total_sales|
+----------+-------------------+
|      1001|7.952609160018033E7|
+----------+-------------------+



In [20]:
#cust_df.select("purchase_date").agg(min("purchase_date"),max("purchase_date"))

min(purchase_date),max(purchase_date)
2023-05-15,2023-06-15


In [23]:
## A.2 Find the top 10 customers with maximum transaction amount for the samedate range of start_date = "2023-05-01" to end_date = "2023-06-08"

In [41]:
cust_df6 = cust_df.filter("purchase_date >= '2023-05-01' and purchase_date <= '2023-06-08' ").select("customer_id","amount").groupBy("customer_id").agg(sum("amount").alias("sum_tran")).sort("sum_tran",ascending=False)

In [43]:
cust_df6.show(10)

+-----------+--------------------+
|customer_id|            sum_tran|
+-----------+--------------------+
|       1001| 3.180884580005336E8|
|       1004| 3.101342580008687E8|
|       1005|2.6240905800151232E8|
|       1003|2.1468385800145328E8|
|       1002| 2.067296580014408E8|
|       1011|1.2724374111049211E8|
|       1006|1.2723851611049213E8|
|       1012|1.1133638611046082E8|
|       1007|1.1133116111046082E8|
|       1013| 9.542903111041903E7|
+-----------+--------------------+
only showing top 10 rows



## using sql

In [31]:
##A.3 Implement all of the above using Spark Table (Create an External Table).

In [ ]:
spark.sql("create table itv024128.cust_transf (customer_id long, purchase_date date, product_id long, amount double) using csv location '/public/trendytech/datasets/cust_transf.csv'")

In [5]:
spark.sql("cache table itv024128.cust_transf")

""


In [6]:
results = spark.sql("select product_id, sum(amount) as sum from  itv024128.cust_transf where purchase_date >= '2023-05-01' and purchase_date <= '2023-06-08' group by product_id order by sum desc limit 10")

In [7]:
results.show()

+----------+--------------------+
|product_id|                 sum|
+----------+--------------------+
|      1003| 5.725592243903786E8|
|      1001|  5.56682641192824E8|
|      1002|4.2938362439486486E8|
|      1004|2.8620802440276194E8|
|      1005| 2.782856412021384E8|
|      1015|  12537.909999999963|
|      1014|  11492.909999999963|
|      1013|  10447.909999999963|
|      1012|   9402.909999999965|
|      1011|   8357.909999999967|
+----------+--------------------+



In [39]:
spark.sql("select customer_id,sum(amount) as sum from itv024128.cust_transf where purchase_date >= '2023-05-01' and purchase_date <= '2023-06-08' group by customer_id order by sum desc").show(10)

+-----------+--------------------+
|customer_id|                 sum|
+-----------+--------------------+
|       1001| 3.180884580005336E8|
|       1004| 3.101342580008686E8|
|       1005| 2.624090580015123E8|
|       1003|2.1468385800145325E8|
|       1002| 2.067296580014408E8|
|       1011| 1.272437411104921E8|
|       1006|1.2723851611049211E8|
|       1012|1.1133638611046082E8|
|       1007|1.1133116111046082E8|
|       1013| 9.542903111041903E7|
+-----------+--------------------+
only showing top 10 rows



In [44]:
## Find the top 10 regular customers based on the number of distinct months in which they made purchases.

### Year and month convert

In [4]:
reg_cust = cust_df.withColumn("purchase_year",year("purchase_date")).withColumn("purchase_month",month("purchase_date"))

In [5]:
reg_cust.show(5)

+-----------+-------------+----------+------+-------------+--------------+
|customer_id|purchase_date|product_id|amount|purchase_year|purchase_month|
+-----------+-------------+----------+------+-------------+--------------+
|       1001|   2023-05-15|      1001| 49.99|         2023|             5|
|       1002|   2023-05-16|      1002| 29.99|         2023|             5|
|       1003|   2023-05-17|      1003| 39.99|         2023|             5|
|       1004|   2023-05-18|      1004| 19.99|         2023|             5|
|       1005|   2023-05-19|      1005| 24.99|         2023|             5|
+-----------+-------------+----------+------+-------------+--------------+
only showing top 5 rows



### countDistinct- agg function

In [7]:
reg_cust1 = reg_cust.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [10]:
reg_cust1.orderBy("distinct_mnths",ascending=False).show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1002|             2|
|       1010|             2|
|       1005|             2|
|       1011|             2|
|       1007|             2|
|       1008|             2|
|       1004|             2|
|       1006|             2|
|       1003|             2|
|       1001|             2|
+-----------+--------------+
only showing top 10 rows



In [ ]:
##reg_cust.unpersist()

In [23]:
reg_cust_cache =reg_cust.cache()

In [24]:
reg_cust1 = reg_cust_cache.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [25]:
reg_cust1.orderBy("distinct_mnths",ascending=False).show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1010|             2|
|       1002|             2|
|       1008|             2|
|       1004|             2|
|       1006|             2|
|       1003|             2|
|       1012|             2|
|       1001|             2|
|       1005|             2|
|       1014|             2|
+-----------+--------------+
only showing top 10 rows



In [5]:
from pyspark import StorageLevel

In [27]:
reg_cust_persist =reg_cust.persist(StorageLevel.MEMORY_AND_DISK)

In [28]:
reg_cust2 = reg_cust_persist.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [29]:
reg_cust2.orderBy("distinct_mnths",ascending=False).show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1012|             2|
|       1005|             2|
|       1015|             2|
|       1013|             2|
|       1007|             2|
|       1011|             2|
|       1009|             2|
|       1006|             2|
|       1004|             2|
|       1003|             2|
+-----------+--------------+
only showing top 10 rows



In [ ]:
reg_cust.unpersist()

In [6]:
reg_cust_persist =reg_cust.persist(StorageLevel.DISK_ONLY)

In [7]:
reg_cust_persist.count()

87498290

In [8]:
reg_cust2 = reg_cust_persist.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [ ]:
reg_cust2.show(10)

In [9]:
reg_cust.unpersist()

customer_id,purchase_date,product_id,amount,purchase_year,purchase_month
1001,2023-05-15,1001,49.99,2023,5
1002,2023-05-16,1002,29.99,2023,5
1003,2023-05-17,1003,39.99,2023,5
1004,2023-05-18,1004,19.99,2023,5
1005,2023-05-19,1005,24.99,2023,5
1001,2023-05-20,1002,29.99,2023,5
1002,2023-05-21,1003,39.99,2023,5
1003,2023-05-22,1004,19.99,2023,5
1004,2023-05-23,1005,24.99,2023,5
1005,2023-05-24,1001,49.99,2023,5


In [10]:
reg_cust_persist =reg_cust.persist(StorageLevel.MEMORY_AND_DISK_DESER)

In [11]:
reg_cust_persist.count()

87498290

In [12]:
reg_cust2 = reg_cust_persist.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [13]:
reg_cust2.show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1010|             2|
|       1002|             2|
|       1012|             2|
|       1009|             2|
|       1013|             2|
|       1007|             2|
|       1011|             2|
|       1005|             2|
|       1001|             2|
|       1015|             2|
+-----------+--------------+
only showing top 10 rows



In [14]:
reg_cust.unpersist()

customer_id,purchase_date,product_id,amount,purchase_year,purchase_month
1001,2023-05-15,1001,49.99,2023,5
1002,2023-05-16,1002,29.99,2023,5
1003,2023-05-17,1003,39.99,2023,5
1004,2023-05-18,1004,19.99,2023,5
1005,2023-05-19,1005,24.99,2023,5
1001,2023-05-20,1002,29.99,2023,5
1002,2023-05-21,1003,39.99,2023,5
1003,2023-05-22,1004,19.99,2023,5
1004,2023-05-23,1005,24.99,2023,5
1005,2023-05-24,1001,49.99,2023,5


In [15]:
reg_cust_persist =reg_cust.persist(StorageLevel.MEMORY_ONLY)

In [16]:
reg_cust_persist.count()

87498290

In [17]:
reg_cust2 = reg_cust_persist.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [18]:
reg_cust2.show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1010|             2|
|       1002|             2|
|       1012|             2|
|       1009|             2|
|       1013|             2|
|       1011|             2|
|       1007|             2|
|       1005|             2|
|       1001|             2|
|       1015|             2|
+-----------+--------------+
only showing top 10 rows



In [19]:
reg_cust.unpersist()

customer_id,purchase_date,product_id,amount,purchase_year,purchase_month
1001,2023-05-15,1001,49.99,2023,5
1002,2023-05-16,1002,29.99,2023,5
1003,2023-05-17,1003,39.99,2023,5
1004,2023-05-18,1004,19.99,2023,5
1005,2023-05-19,1005,24.99,2023,5
1001,2023-05-20,1002,29.99,2023,5
1002,2023-05-21,1003,39.99,2023,5
1003,2023-05-22,1004,19.99,2023,5
1004,2023-05-23,1005,24.99,2023,5
1005,2023-05-24,1001,49.99,2023,5


In [20]:
reg_cust_persist =reg_cust.persist(StorageLevel.MEMORY_AND_DISK)

In [21]:
reg_cust_persist.count()

87498290

In [22]:
reg_cust2 = reg_cust_persist.groupBy("customer_id").agg(countDistinct("purchase_year","purchase_month").alias("distinct_mnths"))

In [23]:
reg_cust2.show(10)

+-----------+--------------+
|customer_id|distinct_mnths|
+-----------+--------------+
|       1010|             2|
|       1002|             2|
|       1012|             2|
|       1009|             2|
|       1013|             2|
|       1007|             2|
|       1011|             2|
|       1005|             2|
|       1001|             2|
|       1015|             2|
+-----------+--------------+
only showing top 10 rows

